In [ ]:
!wget https://raw.githubusercontent.com/Pirogjikdhima/Diploma/refs/heads/main/Corpus/Combined/combined_dataset.csv


--2025-09-18 23:31:55--  https://raw.githubusercontent.com/Pirogjikdhima/Diploma/refs/heads/main/Corpus/Combined/combined_dataset.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26130198 (25M) [text/plain]
Saving to: ‘combined_dataset.csv’

combined_dataset.cs 100%[===================>]  24.92M  --.-KB/s    in 0.07s   

2025-09-18 23:31:57 (361 MB/s) - ‘combined_dataset.csv’ saved [26130198/26130198]



In [ ]:
!pip install transformers
!pip install tokenizers
!pip install accelerate
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=d2ec4c7c20547e5ce798f3f479362f38a7b1a3f39d3a840069c68965f3dd05fd
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [ ]:
import torch
import numpy as np
import pandas as pd
import time
from datetime import datetime
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from seqeval.metrics import f1_score as seq_f1_score
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification, XLMRobertaForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification,TrainerCallback
)
from datasets import Dataset
from sklearn.model_selection import train_test_split

In [ ]:
dataset = pd.read_csv("combined_dataset.csv")

In [ ]:
class BERT:
    def __init__(self, dataset, model_name="bert-base-multilingual-cased"):
        """
        Initialize BERT NER model
        """
        self.model_name = model_name
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.dataset = dataset
        self.max_length = 128
        self.training_time = 0

        # Get labels and sentences
        self.label2id, self.id2label = self.get_labels()
        self.num_labels = len(self.label2id)
        self.sentences = self.get_sentences()

        # Initialize tokenizer and model
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(
            self.model_name,
            num_labels=self.num_labels,
            id2label=self.id2label,
            label2id=self.label2id,
            ignore_mismatched_sizes=True
        ).to(self.device)

        self.data_collator = DataCollatorForTokenClassification(
            tokenizer=self.tokenizer,
            padding=True,
            return_tensors="pt"
        )

        self.train_dataset = None
        self.val_dataset = None
        self.test_dataset = None
        self.trainer = None
        self.training_losses = []

        print(f"🤖 BERT Model Initialized: {self.model_name}")

    def get_labels(self):
        """Extract unique labels and create mapping dictionaries"""
        tags = sorted(list(set(self.dataset["NER_TAG"].values)))
        label2id = {tag: i for i, tag in enumerate(tags)}
        id2label = {v: k for k, v in label2id.items()}
        return label2id, id2label

    def get_sentences(self):
        """Convert dataset to list of (word, tag) tuples grouped by sentence"""
        def to_tuples(group):
            return list(zip(group["WORD"].values, group["NER_TAG"].values))

        sentences = self.dataset.groupby("SENTENCE #").apply(
            to_tuples, include_groups=False
        ).tolist()
        return sentences

    def align_labels_with_tokens(self, words, labels):
        """Align labels with BERT subword tokens"""
        tokenized_inputs = self.tokenizer(
            words,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            is_split_into_words=True,
            return_offsets_mapping=True
        )

        aligned_labels = []
        word_ids = tokenized_inputs.word_ids()
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                aligned_labels.append(-100)
            elif word_idx != previous_word_idx:
                aligned_labels.append(labels[word_idx])
            else:
                aligned_labels.append(-100)
            previous_word_idx = word_idx

        return {
            'input_ids': tokenized_inputs['input_ids'],
            'attention_mask': tokenized_inputs['attention_mask'],
            'labels': aligned_labels
        }

    def prepare_dataset(self, sentences, labels):
        """Prepare dataset for BERT training"""
        all_input_ids = []
        all_attention_masks = []
        all_labels = []

        for words, sentence_labels in zip(sentences, labels):
            if len(words) == 0 or len(sentence_labels) == 0 or len(words) != len(sentence_labels):
                continue

            aligned_data = self.align_labels_with_tokens(words, sentence_labels)
            all_input_ids.append(aligned_data['input_ids'])
            all_attention_masks.append(aligned_data['attention_mask'])
            all_labels.append(aligned_data['labels'])

        return Dataset.from_dict({
            'input_ids': all_input_ids,
            'attention_mask': all_attention_masks,
            'labels': all_labels
        })

    def prepare_data_splits(self, test_size=0.1, val_size=0.1):
        """Prepare train/validation/test splits"""
        words = []
        tag_indices = []

        for sentence in self.sentences:
            sentence_words = [word for word, tag in sentence]
            sentence_tags = [self.label2id[tag] for word, tag in sentence]
            words.append(sentence_words)
            tag_indices.append(sentence_tags)

        # Create splits
        words_temp, words_test, tags_temp, tags_test = train_test_split(
            words, tag_indices, test_size=test_size, random_state=42
        )

        val_size_adjusted = val_size / (1 - test_size)
        words_train, words_val, tags_train, tags_val = train_test_split(
            words_temp, tags_temp, test_size=val_size_adjusted, random_state=42
        )

        # Create datasets
        self.train_dataset = self.prepare_dataset(words_train, tags_train)
        self.val_dataset = self.prepare_dataset(words_val, tags_val)
        self.test_dataset = self.prepare_dataset(words_test, tags_test)

        return self.train_dataset, self.val_dataset, self.test_dataset

    def compute_metrics(self, eval_pred):
        """Compute token-level (micro) F1 and entity-level (seqeval) F1"""
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=2)

        # Remove ignored indices (-100) for token-level metrics
        true_predictions = []
        true_labels = []

        # Also prepare for seqeval (entity-level)
        seqeval_predictions = []
        seqeval_labels = []

        for prediction, label in zip(predictions, labels):
            # Token-level: filter out -100
            pred_tokens = [p for (p, l) in zip(prediction, label) if l != -100]
            true_tokens = [l for (p, l) in zip(prediction, label) if l != -100]

            if len(pred_tokens) > 0:
                true_predictions.extend(pred_tokens)
                true_labels.extend(true_tokens)

                # Convert to labels for seqeval
                pred_labels = [self.id2label[p] for p in pred_tokens]
                true_labels_seq = [self.id2label[l] for l in true_tokens]

                seqeval_predictions.append(pred_labels)
                seqeval_labels.append(true_labels_seq)

        # Token-level F1 (micro average)
        _, _, token_f1, _ = precision_recall_fscore_support(
            true_labels, true_predictions, average='micro', zero_division=0
        )

        # Entity-level F1 using seqeval
        entity_f1 = seq_f1_score(seqeval_labels, seqeval_predictions)

        return {
            "token_f1": float(token_f1),
            "entity_f1": float(entity_f1),
        }

    def setup_trainer(self, output_dir='./bert_ner_results', num_epochs=3,
                     train_batch_size=32, eval_batch_size=64):
        """Setup HuggingFace Trainer"""
        if self.train_dataset is None:
            print("❌ Run prepare_data_splits() first.")
            return None

        # Custom callback to track losses
        class LossCallback(TrainerCallback):
            def __init__(self, roberta_instance):
                self.roberta_instance = roberta_instance

            def on_log(self, args, state, control, model=None, logs=None, **kwargs):
                if logs and 'train_loss' in logs:
                    self.roberta_instance.training_losses.append(logs['train_loss'])

        training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=num_epochs,
            per_device_train_batch_size=train_batch_size,
            per_device_eval_batch_size=eval_batch_size,
            warmup_steps=500,
            weight_decay=0.01,
            learning_rate=2e-5,
            logging_steps=100,
            eval_strategy="steps",
            eval_steps=200,
            save_steps=400,
            save_total_limit=2,
            load_best_model_at_end=True,
            metric_for_best_model="entity_f1",
            greater_is_better=True,
            report_to=[],
            seed=42,
            fp16=torch.cuda.is_available(),
            remove_unused_columns=False,
            push_to_hub=False,
            gradient_accumulation_steps=2,
            max_grad_norm=1.0,
            lr_scheduler_type="linear",
        )

        self.trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=self.train_dataset,
            eval_dataset=self.val_dataset,
            data_collator=self.data_collator,
            tokenizer=self.tokenizer,
            compute_metrics=self.compute_metrics,
            callbacks=[LossCallback(self)]
        )

        print(f"🎯 BERT Trainer setup complete!")
        return self.trainer

    def train_and_evaluate(self, save_model_path='./best_bert_model'):
        """Train model and return metrics dictionary"""
        if self.trainer is None:
            print("❌ Run setup_trainer() first.")
            return None

        print("🚀 STARTING BERT TRAINING")
        start_time = time.time()

        try:
            # Train
            self.trainer.train()

            # Calculate training time
            end_time = time.time()
            self.training_time = end_time - start_time

            # Evaluate on test set
            eval_results = self.trainer.evaluate(self.test_dataset)

            # Save model
            self.trainer.save_model(save_model_path)

            print(f"✅ BERT Training completed in {self.training_time:.5f} seconds")

            # Return results dictionary
            results = {
                'Model': 'BERT',
                'Token F1': eval_results['eval_token_f1'],
                'Entity F1': eval_results['eval_entity_f1'],
                'Training Time': self.training_time
            }

            return results

        except Exception as e:
            print(f"❌ Training failed: {e}")
            return None

    def get_training_losses(self):
        """Return training losses for plotting"""
        return self.training_losses

In [ ]:
class RoBERTa:
    def __init__(self, dataset, model_name="xlm-roberta-base"):
        """
        Initialize RoBERTa NER model
        """
        self.model_name = model_name
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.dataset = dataset
        self.max_length = 128
        self.training_time = 0

        # Get labels and sentences
        self.label2id, self.id2label = self.get_labels()
        self.num_labels = len(self.label2id)
        self.sentences = self.get_sentences()

        # Initialize tokenizer and model
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = XLMRobertaForTokenClassification.from_pretrained(
            self.model_name,
            num_labels=self.num_labels,
            id2label=self.id2label,
            label2id=self.label2id,
            ignore_mismatched_sizes=True
        ).to(self.device)

        self.data_collator = DataCollatorForTokenClassification(
            tokenizer=self.tokenizer,
            padding=True,
            return_tensors="pt"
        )

        self.train_dataset = None
        self.val_dataset = None
        self.test_dataset = None
        self.trainer = None
        self.training_losses = []

        print(f"🤖 RoBERTa Model Initialized: {self.model_name}")

    def get_labels(self):
        """Extract unique labels and create mapping dictionaries"""
        tags = sorted(list(set(self.dataset["NER_TAG"].values)))
        label2id = {tag: i for i, tag in enumerate(tags)}
        id2label = {v: k for k, v in label2id.items()}
        return label2id, id2label

    def get_sentences(self):
        """Convert dataset to list of (word, tag) tuples grouped by sentence"""
        def to_tuples(group):
            return list(zip(group["WORD"].values, group["NER_TAG"].values))

        sentences = self.dataset.groupby("SENTENCE #").apply(
            to_tuples, include_groups=False
        ).tolist()
        return sentences

    def align_labels_with_tokens(self, words, labels):
        """Align labels with RoBERTa subword tokens"""
        tokenized_inputs = self.tokenizer(
            words,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            is_split_into_words=True,
            return_offsets_mapping=True
        )

        aligned_labels = []
        word_ids = tokenized_inputs.word_ids()
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                aligned_labels.append(-100)
            elif word_idx != previous_word_idx:
                aligned_labels.append(labels[word_idx])
            else:
                aligned_labels.append(-100)
            previous_word_idx = word_idx

        return {
            'input_ids': tokenized_inputs['input_ids'],
            'attention_mask': tokenized_inputs['attention_mask'],
            'labels': aligned_labels
        }

    def prepare_dataset(self, sentences, labels):
        """Prepare dataset for RoBERTa training"""
        all_input_ids = []
        all_attention_masks = []
        all_labels = []

        for words, sentence_labels in zip(sentences, labels):
            if len(words) == 0 or len(sentence_labels) == 0 or len(words) != len(sentence_labels):
                continue

            aligned_data = self.align_labels_with_tokens(words, sentence_labels)
            all_input_ids.append(aligned_data['input_ids'])
            all_attention_masks.append(aligned_data['attention_mask'])
            all_labels.append(aligned_data['labels'])

        return Dataset.from_dict({
            'input_ids': all_input_ids,
            'attention_mask': all_attention_masks,
            'labels': all_labels
        })

    def prepare_data_splits(self, test_size=0.1, val_size=0.1):
        """Prepare train/validation/test splits"""
        words = []
        tag_indices = []

        for sentence in self.sentences:
            sentence_words = [word for word, tag in sentence]
            sentence_tags = [self.label2id[tag] for word, tag in sentence]
            words.append(sentence_words)
            tag_indices.append(sentence_tags)

        # Create splits
        words_temp, words_test, tags_temp, tags_test = train_test_split(
            words, tag_indices, test_size=test_size, random_state=42
        )

        val_size_adjusted = val_size / (1 - test_size)
        words_train, words_val, tags_train, tags_val = train_test_split(
            words_temp, tags_temp, test_size=val_size_adjusted, random_state=42
        )

        # Create datasets
        self.train_dataset = self.prepare_dataset(words_train, tags_train)
        self.val_dataset = self.prepare_dataset(words_val, tags_val)
        self.test_dataset = self.prepare_dataset(words_test, tags_test)

        return self.train_dataset, self.val_dataset, self.test_dataset

    def compute_metrics(self, eval_pred):
        """Compute token-level (micro) F1 and entity-level (seqeval) F1"""
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=2)

        # Remove ignored indices (-100) for token-level metrics
        true_predictions = []
        true_labels = []

        # Also prepare for seqeval (entity-level)
        seqeval_predictions = []
        seqeval_labels = []

        for prediction, label in zip(predictions, labels):
            # Token-level: filter out -100
            pred_tokens = [p for (p, l) in zip(prediction, label) if l != -100]
            true_tokens = [l for (p, l) in zip(prediction, label) if l != -100]

            if len(pred_tokens) > 0:
                true_predictions.extend(pred_tokens)
                true_labels.extend(true_tokens)

                # Convert to labels for seqeval
                pred_labels = [self.id2label[p] for p in pred_tokens]
                true_labels_seq = [self.id2label[l] for l in true_tokens]

                seqeval_predictions.append(pred_labels)
                seqeval_labels.append(true_labels_seq)

        # Token-level F1 (micro average)
        _, _, token_f1, _ = precision_recall_fscore_support(
            true_labels, true_predictions, average='micro', zero_division=0
        )

        # Entity-level F1 using seqeval
        entity_f1 = seq_f1_score(seqeval_labels, seqeval_predictions)

        return {
            "token_f1": float(token_f1),
            "entity_f1": float(entity_f1),
        }

    def setup_trainer(self, output_dir='./roberta_ner_results', num_epochs=4,
                     train_batch_size=32, eval_batch_size=64):
        """Setup HuggingFace Trainer for RoBERTa"""
        if self.train_dataset is None:
            print("❌ Run prepare_data_splits() first.")
            return None

        # Custom callback to track losses
        class LossCallback(TrainerCallback):
            def __init__(self, roberta_instance):
                self.roberta_instance = roberta_instance

            def on_log(self, args, state, control, model=None, logs=None, **kwargs):
                if logs and 'train_loss' in logs:
                    self.roberta_instance.training_losses.append(logs['train_loss'])

        training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=num_epochs,
            per_device_train_batch_size=train_batch_size,
            per_device_eval_batch_size=eval_batch_size,
            warmup_steps=500,
            weight_decay=0.01,
            learning_rate=2e-5,
            logging_steps=100,
            eval_strategy="steps",
            eval_steps=200,
            save_steps=400,
            save_total_limit=2,
            load_best_model_at_end=True,
            metric_for_best_model="entity_f1",
            greater_is_better=True,
            report_to=[],
            seed=42,
            fp16=torch.cuda.is_available(),
            remove_unused_columns=False,
            push_to_hub=False,
            gradient_accumulation_steps=2,
            max_grad_norm=1.0,
            lr_scheduler_type="linear",
        )

        self.trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=self.train_dataset,
            eval_dataset=self.val_dataset,
            data_collator=self.data_collator,
            tokenizer=self.tokenizer,
            compute_metrics=self.compute_metrics,
            callbacks=[LossCallback(self)]
        )

        print(f"🎯 RoBERTa Trainer setup complete!")
        return self.trainer

    def train_and_evaluate(self, save_model_path='./best_roberta_model'):
        """Train model and return metrics dictionary"""
        if self.trainer is None:
            print("❌ Run setup_trainer() first.")
            return None

        print("🚀 STARTING RoBERTa TRAINING")
        start_time = time.time()

        try:
            # Train
            self.trainer.train()

            # Calculate training time
            end_time = time.time()
            self.training_time = end_time - start_time

            # Evaluate on test set
            eval_results = self.trainer.evaluate(self.test_dataset)

            # Save model
            self.trainer.save_model(save_model_path)

            print(f"✅ RoBERTa Training completed in {self.training_time:.2f} seconds")

            # Return results dictionary
            results = {
                'Model': 'RoBERTa',
                'Token F1': eval_results['eval_token_f1'],
                'Entity F1': eval_results['eval_entity_f1'],
                'Training Time': self.training_time
            }

            return results

        except Exception as e:
            print(f"❌ Training failed: {e}")
            return None

    def get_training_losses(self):
        """Return training losses for plotting"""
        return self.training_losses

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

def create_results_dataframe(bert_results, roberta_results):
    """
    Create a DataFrame from BERT and RoBERTa results

    Args:
        bert_results: Dictionary with BERT results
        roberta_results: Dictionary with RoBERTa results

    Returns:
        pandas DataFrame with comparison results
    """
    # Combine results
    all_results = [bert_results, roberta_results]

    # Create DataFrame
    df = pd.DataFrame(all_results)

    # Round numerical values for better display
    df['Token F1'] = df['Token F1'].round(4)
    df['Entity F1'] = df['Entity F1'].round(4)
    df['Training Time'] = df['Training Time'].round(2)

    print("📊 Model Comparison Results:")
    print("=" * 60)
    print(df.to_string(index=False))
    print("=" * 60)

    return df

def print_detailed_comparison(df):
    """
    Print detailed comparison analysis
    """
    print("\n🔍 DETAILED ANALYSIS")
    print("=" * 50)

    bert_row = df[df['Model'] == 'BERT'].iloc[0]
    roberta_row = df[df['Model'] == 'RoBERTa'].iloc[0]

    # Token F1 comparison
    token_diff = roberta_row['Token F1'] - bert_row['Token F1']
    print(f"Token F1 Scores:")
    print(f"  BERT:    {bert_row['Token F1']:.4f}")
    print(f"  RoBERTa: {roberta_row['Token F1']:.4f}")
    print(f"  Difference: {token_diff:+.4f} ({'RoBERTa wins' if token_diff > 0 else 'BERT wins' if token_diff < 0 else 'Tie'})")

    # Entity F1 comparison
    entity_diff = roberta_row['Entity F1'] - bert_row['Entity F1']
    print(f"\nEntity F1 Scores:")
    print(f"  BERT:    {bert_row['Entity F1']:.4f}")
    print(f"  RoBERTa: {roberta_row['Entity F1']:.4f}")
    print(f"  Difference: {entity_diff:+.4f} ({'RoBERTa wins' if entity_diff > 0 else 'BERT wins' if entity_diff < 0 else 'Tie'})")

    # Training time comparison
    time_diff = roberta_row['Training Time'] - bert_row['Training Time']
    print(f"\nTraining Time:")
    print(f"  BERT:    {bert_row['Training Time']:.5f} seconds")
    print(f"  RoBERTa: {roberta_row['Training Time']:.5f} seconds")
    print(f"  Difference: {time_diff:+.2f} seconds ({'BERT faster' if time_diff > 0 else 'RoBERTa faster' if time_diff < 0 else 'Same speed'})")

    # Overall winner
    print(f"\n🏆 OVERALL ASSESSMENT:")

    bert_score = 0
    roberta_score = 0

    if token_diff > 0:
        roberta_score += 1
        print(f"  Token F1: RoBERTa wins (+{token_diff:.4f})")
    elif token_diff < 0:
        bert_score += 1
        print(f"  Token F1: BERT wins ({token_diff:.4f})")
    else:
        print(f"  Token F1: Tie")

    if entity_diff > 0:
        roberta_score += 1
        print(f"  Entity F1: RoBERTa wins (+{entity_diff:.4f})")
    elif entity_diff < 0:
        bert_score += 1
        print(f"  Entity F1: BERT wins ({entity_diff:.4f})")
    else:
        print(f"  Entity F1: Tie")

    if time_diff < 0:  # Less time is better
        roberta_score += 1
        print(f"  Speed: RoBERTa wins ({-time_diff:.2f} min faster)")
    elif time_diff > 0:
        bert_score += 1
        print(f"  Speed: BERT wins ({time_diff:.2f} min faster)")
    else:
        print(f"  Speed: Tie")

    if bert_score > roberta_score:
        winner = "BERT"
    elif roberta_score > bert_score:
        winner = "RoBERTa"
    else:
        winner = "Tie"

    print(f"\n🎯 FINAL WINNER: {winner}")
    print(f"   BERT Score: {bert_score}/3")
    print(f"   RoBERTa Score: {roberta_score}/3")

# Example usage function for Colab
def run_complete_comparison(dataset):
    """
    Complete workflow for comparing BERT and RoBERTa models

    Args:
        dataset: pandas DataFrame with NER data

    Returns:
        DataFrame with comparison results
    """
    print("🚀 STARTING COMPLETE MODEL COMPARISON")
    print("=" * 60)

    # Initialize models
    bert_model = BERT(dataset)
    roberta_model = RoBERTa(dataset)

    # Prepare data for both models
    print("\n📊 Preparing BERT data...")
    bert_model.prepare_data_splits()
    bert_model.setup_trainer()

    print("\n📊 Preparing RoBERTa data...")
    roberta_model.prepare_data_splits()
    roberta_model.setup_trainer()

    # Train and evaluate BERT
    print("\n" + "="*60)
    bert_results = bert_model.train_and_evaluate()
    bert_losses = bert_model.get_training_losses()

    # Train and evaluate RoBERTa
    print("\n" + "="*60)
    roberta_results = roberta_model.train_and_evaluate()
    roberta_losses = roberta_model.get_training_losses()

    # Create comparison DataFrame
    comparison_df = create_results_dataframe(bert_results, roberta_results)

    # Generate plots and analysis
    plot_model_comparison(comparison_df, bert_losses, roberta_losses)
    print_detailed_comparison(comparison_df)

    return comparison_df

In [ ]:
# Initialize and train BERT
print("🤖 Initializing BERT Model...")
bert_model = BERT(dataset)

# Prepare data
print("📊 Preparing BERT data splits...")
bert_model.prepare_data_splits()

# Setup trainer
print("🎯 Setting up BERT trainer...")
bert_model.setup_trainer(num_epochs=5, train_batch_size=32)

# Train and evaluate
print("🚀 Training BERT model...")
bert_results = bert_model.train_and_evaluate()
bert_losses = bert_model.get_training_losses()

print(f"✅ BERT Results: {bert_results}")

# ===== CELL 4: RoBERTa Model Training =====
# Paste the RoBERTa class code here (from the second artifact)

# Initialize and train RoBERTa
print("🤖 Initializing RoBERTa Model...")
roberta_model = RoBERTa(dataset)

# Prepare data
print("📊 Preparing RoBERTa data splits...")
roberta_model.prepare_data_splits()

# Setup trainer
print("🎯 Setting up RoBERTa trainer...")
roberta_model.setup_trainer(num_epochs=5, train_batch_size=32)

# Train and evaluate
print("🚀 Training RoBERTa model...")
roberta_results = roberta_model.train_and_evaluate()
roberta_losses = roberta_model.get_training_losses()

print(f"✅ RoBERTa Results: {roberta_results}")



🤖 Initializing BERT Model...


Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🤖 BERT Model Initialized: bert-base-multilingual-cased
📊 Preparing BERT data splits...
🎯 Setting up BERT trainer...
🎯 BERT Trainer setup complete!
🚀 Training BERT model...
🚀 STARTING BERT TRAINING


/tmp/ipython-input-4134253961.py:221: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  self.trainer = Trainer(


Step,Training Loss,Validation Loss,Token F1,Entity F1
200,0.209300,0.103944,0.973471,0.786799
400,0.086000,0.067540,0.981568,0.865640
600,0.060900,0.054240,0.984102,0.885800
800,0.054500,0.050991,0.984667,0.892549
1000,0.048100,0.045779,0.986577,0.901318
1200,0.039200,0.046653,0.986438,0.905382
1400,0.035900,0.044128,0.987399,0.909554
1600,0.031400,0.044059,0.987725,0.914364
1800,0.032100,0.044005,0.987369,0.913340
2000,0.026400,0.044991,0.987577,0.915368


...

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🤖 RoBERTa Model Initialized: xlm-roberta-base
📊 Preparing RoBERTa data splits...
🎯 Setting up RoBERTa trainer...
🎯 RoBERTa Trainer setup complete!
🚀 Training RoBERTa model...
🚀 STARTING RoBERTa TRAINING


/tmp/ipython-input-2095020884.py:221: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  self.trainer = Trainer(


Step,Training Loss,Validation Loss,Token F1,Entity F1
200,0.349700,0.203244,0.946519,0.407451
400,0.098200,0.072048,0.980480,0.847440
600,0.066200,0.054687,0.984228,0.890756
800,0.058500,0.050960,0.985461,0.897886
1000,0.049200,0.045549,0.986181,0.901549
1200,0.042000,0.044850,0.986437,0.906508
1400,0.040400,0.043475,0.987572,0.914053
1600,0.035900,0.043309,0.987424,0.913292
1800,0.036600,0.042339,0.987611,0.914644
2000,0.031000,0.043659,0.987552,0.918015


✅ RoBERTa Training completed in 336.85 seconds
✅ RoBERTa Results: {'Model': 'RoBERTa', 'Token F1': 0.9859619417016962, 'Entity F1': 0.9060190073917636, 'Training Time': 336.8548107147217}


In [ ]:
print("📊 Creating comparison DataFrame...")
comparison_df = create_results_dataframe(bert_results, roberta_results)

comparison_df.to_csv('model_comparison_results.csv', index=False)
print("💾 Results saved to 'model_comparison_results.csv'")

print_detailed_comparison(comparison_df)

📊 Creating comparison DataFrame...
📊 Model Comparison Results:
  Model  Token F1  Entity F1  Training Time
   BERT    0.9861     0.9074         305.34
RoBERTa    0.9860     0.9060         336.85
💾 Results saved to 'model_comparison_results.csv'

🔍 DETAILED ANALYSIS
Token F1 Scores:
  BERT:    0.9861
  RoBERTa: 0.9860
  Difference: -0.0001 (BERT wins)

Entity F1 Scores:
  BERT:    0.9074
  RoBERTa: 0.9060
  Difference: -0.0014 (BERT wins)

Training Time:
  BERT:    305.34000 seconds
  RoBERTa: 336.85000 seconds
  Difference: +31.51 seconds (BERT faster)

🏆 OVERALL ASSESSMENT:
  Token F1: BERT wins (-0.0001)
  Entity F1: BERT wins (-0.0014)
  Speed: BERT wins (31.51 min faster)

🎯 FINAL WINNER: BERT
   BERT Score: 3/3
   RoBERTa Score: 0/3
